In [1]:
import os
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import boto3

#### Functions

In [2]:
def get_tier(flt_ecnl, dict_tiers):
    if flt_ecnl <= dict_tiers['A1']:
        return 'A1'
    elif flt_ecnl <= dict_tiers['A']:
        return 'A'
    elif flt_ecnl <= dict_tiers['B']:
        return 'B'
    elif flt_ecnl <= dict_tiers['C']:
        return 'C'
    elif flt_ecnl <= dict_tiers['D']:
        return 'D'
    else:
        return 'Decline'

#### Constants

In [3]:
str_dirname_output = './output'
str_uri = 's3://20250121-gen-13-model-monitoring/07_compare_debtor/df.gzip'

# dict tiers
dict_tiers = {
    'A1': 0.0760,
    'A': 0.1320,
    'B': 0.2650,
    'C': 0.3220,
    'D': 0.3500, 
}

# factor 24 to 72
flt_factor_24_to_72 = 2.36

# pd threshold
flt_threshold_pd = 0.6

#### Output directory

In [4]:
try:
    os.mkdir(str_dirname_output)
except:
    pass

#### Read in data

In [5]:
# pull data
df = pd.read_parquet(str_uri)
# impute repo as 5 (max)
df['rp01s__tu'] = df['rp01s__tu'].fillna(4)
# wtd avg impute to 0
df['ENG-wtd_avg'] = df['ENG-wtd_avg'].fillna(0)
# g232s impute to 383 (max)
df['g232s__tu'] = df['g232s__tu'].fillna(383)

# group by
df = df.groupby('accountid').agg({
    'pd_gen13': 'mean', 
    'lgd_gen13': 'mean',
    'g232s__tu': 'mean',
    'ENG-wtd_avg': 'mean',
    'rp01s__tu': 'mean',
}).reset_index()

# calculate ecnl_gen13
df['ecnl_gen13'] = df['pd_gen13'] * df['lgd_gen13'] * flt_factor_24_to_72

# get tier
df['tier_gen13'] = df['ecnl_gen13'].apply(
    lambda x: get_tier(
        flt_ecnl=x,
        dict_tiers=dict_tiers,
    ),
)

# remove declines
df = df[df['tier_gen13'] != 'Decline']

# get descriptive stats
df_descriptions = pd.DataFrame({
    'mean': df[['g232s__tu', 'rp01s__tu']].mean(),    
    'min': df[['g232s__tu', 'rp01s__tu']].min(),
    'max': df[['g232s__tu', 'rp01s__tu']].max(),
    'median': df[['g232s__tu', 'rp01s__tu']].median(),
    'mode': df[['g232s__tu', 'rp01s__tu']].mode().iloc[0],
    'sd': df[['g232s__tu', 'rp01s__tu']].std()
})

# 2 std
df_descriptions['2sd'] = df_descriptions['sd'] * 2

# calc upper limit
df_descriptions['upper_limit'] = df_descriptions['2sd'] + df_descriptions['mean']

# show
df_descriptions

,mean,min,max,median,mode,sd,2sd,upper_limit
g232s__tu,12.192604,0.0,383.0,8.5,6.0,17.416377,34.832754,47.025358
rp01s__tu,0.123893,-4.0,4.0,0.0,0.0,0.565669,1.131337,1.255230


#### Do it for non BK for ENG-wtd_avg

In [6]:
# pull data
df = pd.read_parquet(str_uri)
# impute repo as 5 (max)
df['rp01s__tu'] = df['rp01s__tu'].fillna(4)
# wtd avg impute to 0
df['ENG-wtd_avg'] = df['ENG-wtd_avg'].fillna(0)
# g232s impute to 383 (max)
df['g232s__tu'] = df['g232s__tu'].fillna(383)

# subset non BK
df = df[df['intopenbktype__app'].isna()]

# group by
df = df.groupby('accountid').agg({
    'pd_gen13': 'mean', 
    'lgd_gen13': 'mean',
    'g232s__tu': 'mean',
    'ENG-wtd_avg': 'mean',
    'rp01s__tu': 'mean',
}).reset_index()

# calculate ecnl_gen13
df['ecnl_gen13'] = df['pd_gen13'] * df['lgd_gen13'] * flt_factor_24_to_72

# get tier
df['tier_gen13'] = df['ecnl_gen13'].apply(
    lambda x: get_tier(
        flt_ecnl=x,
        dict_tiers=dict_tiers,
    ),
)

# remove declines
df = df[df['tier_gen13'] != 'Decline']

# get descriptive stats
df_descriptions = pd.DataFrame({
    'mean': df[['ENG-wtd_avg']].mean(),    
    'min': df[['ENG-wtd_avg']].min(),
    'max': df[['ENG-wtd_avg']].max(),
    'median': df[['ENG-wtd_avg']].median(),
    'mode': df[['ENG-wtd_avg']].mode().iloc[0],
    '25%': df[['ENG-wtd_avg']].apply(lambda x:np.percentile(x, 25))  ,  
    'sd': df[['ENG-wtd_avg']].std()
})

# 2 std
df_descriptions['2sd'] = df_descriptions['sd'] * 2

# calc upper limit
df_descriptions['upper_limit'] = df_descriptions['2sd'] + df_descriptions['mean']

# show
df_descriptions

,mean,min,max,median,mode,25%,sd,2sd,upper_limit
ENG-wtd_avg,0.576415,0.0,1.0,0.68553,1.0,0.083333,0.411394,0.822788,1.399203
